In [ ]:
import wikipedia
import numpy as np

from langchain.text_splitter import RecursiveCharacterTextSplitter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
topic = "Artificial intelligence"

text = wikipedia.page(topic).content

print("Total Characters:", len(text))
print("\n")
print(text[:1000])

In [ ]:
def fixed_size_chunking(text, chunk_size):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i + chunk_size]

        chunks.append(chunk)

    return chunks

In [ ]:
manual_chunks = fixed_size_chunking(text, 300)

print("Total Chunks:", len(manual_chunks))
print("\nFirst Chunk:\n")
print(manual_chunks[0])

In [ ]:
def recursive_chunking(text, chunk_size, overlap):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )

    chunks = splitter.split_text(text)

    return chunks

In [ ]:
recursive_chunks = recursive_chunking(text, 300, 50)

print("Total Chunks:", len(recursive_chunks))
print("\nFirst Chunk:\n")
print(recursive_chunks[0])

In [ ]:
def retrieve_best_chunk(chunks, query):

    vectorizer = TfidfVectorizer()

    vectors = vectorizer.fit_transform(chunks + [query])

    similarity = cosine_similarity(
        vectors[-1],
        vectors[:-1]
    )

    best_index = similarity.argmax()

    return chunks[best_index], similarity[0][best_index]

In [ ]:
query = "How do machines learn from data?"

In [ ]:
configs = [
    (100, 0),
    (100, 50),
    (100, 100),

    (300, 0),
    (300, 50),
    (300, 100),

    (500, 0),
    (500, 50),
    (500, 100),

    (1000, 0),
    (1000, 50),
    (1000, 100)
]

In [ ]:
results = []

for chunk_size, overlap in configs:

    chunks = recursive_chunking(
        text,
        chunk_size,
        overlap
    )

    best_chunk, score = retrieve_best_chunk(
        chunks,
        query
    )

    results.append({
        "chunk_size": chunk_size,
        "overlap": overlap,
        "score": score,
        "chunk": best_chunk
    })

    print("=" * 80)
    print(f"Chunk Size: {chunk_size}")
    print(f"Overlap: {overlap}")
    print(f"Similarity Score: {score:.4f}")

    print("\nRetrieved Chunk:\n")
    print(best_chunk[:500])